# CloudSQL PostgreSQL Census Data Setup

This notebook demonstrates how to:
1. Create a CloudSQL PostgreSQL instance
2. Set up a database and table schema
3. Load UK Census 2021 demographic data (TS001 dataset)
4. Validate and query the data

**Dataset**: Census 2021 TS001 - "Number of usual residents in households and communal establishments"

**Data Coverage**: 7 geographic levels totaling ~232,000 rows:
- Output Areas (OA): ~188k rows
- Lower Super Output Areas (LSOA): ~35k rows
- Middle Super Output Areas (MSOA): ~7k rows
- Lower Tier Local Authorities (LTLA): ~332 rows
- Upper Tier Local Authorities (UTLA): ~175 rows
- Regions (RGN): ~11 rows
- Countries (CTRY): ~4 rows

**Estimated Time**: 20-25 minutes (including CloudSQL instance provisioning which takes 10-15 minutes)

**Required Permissions**:
- `roles/cloudsql.admin` (to create and manage CloudSQL instances)
- `roles/compute.networkUser` (for network configuration)

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
INSTANCE_NAME = "census-demo-db"
DATABASE_NAME = "census_data"
DB_USER = "postgres"
DB_PASSWORD = "Census2021Demo!"  # Change this to a secure password

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Instance Name: {INSTANCE_NAME}")
print(f"  Database Name: {DATABASE_NAME}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [3]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = [
    "cloud-sql-python-connector",
    "pg8000",
    "sqlalchemy",
    "pandas",
    "google-cloud-resource-manager",
    "certifi"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Libraries installed



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Import libraries
import pandas as pd
import time
import os
import ssl
import certifi
from pathlib import Path
from google.cloud.sql.connector import Connector
import sqlalchemy
from sqlalchemy import text
import requests
from google.auth.transport.requests import Request
import aiohttp

# Configure SSL to use certifi's certificate bundle
# This fixes SSL certificate verification issues on macOS
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Create SSL context for aiohttp (used by Cloud SQL Connector)
ssl_context = ssl.create_default_context(cafile=certifi.where())

# Monkey-patch aiohttp's TCPConnector to use our SSL context
# This is necessary because the Cloud SQL Connector uses aiohttp internally
original_tcp_connector_init = aiohttp.TCPConnector.__init__

def patched_tcp_connector_init(self, *args, **kwargs):
    if 'ssl' not in kwargs or kwargs['ssl'] is None:
        kwargs['ssl'] = ssl_context
    original_tcp_connector_init(self, *args, **kwargs)

aiohttp.TCPConnector.__init__ = patched_tcp_connector_init

# Get credentials for REST API calls
credentials, _ = default()
credentials.refresh(Request())

print("✅ Libraries imported and credentials refreshed")
print(f"✅ SSL configured with certificate bundle: {certifi.where()}")
print("✅ aiohttp patched to use certifi certificates")

✅ Libraries imported and credentials refreshed
✅ SSL configured with certificate bundle: /Users/johnswain/Development/gcp-demo-dataplex/venv/lib/python3.11/site-packages/certifi/cacert.pem
✅ aiohttp patched to use certifi certificates


## Section 2: Create CloudSQL PostgreSQL Instance

This section creates a CloudSQL PostgreSQL instance using the Cloud SQL Admin API.

**Instance Configuration:**
- Tier: db-f1-micro (0.6GB RAM, 3 GB storage)
- Version: PostgreSQL 15
- Public IP enabled for demo access
- Automated backups enabled

**Note**: Instance creation takes approximately 10-15 minutes.

In [5]:
# Check if instance already exists
print("🔍 Checking if CloudSQL instance exists...")

url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances/{INSTANCE_NAME}"
headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    instance_info = response.json()
    print(f"✅ Instance '{INSTANCE_NAME}' already exists")
    print(f"   State: {instance_info.get('state', 'UNKNOWN')}")
    print(f"   Version: {instance_info.get('databaseVersion', 'UNKNOWN')}")
    INSTANCE_EXISTS = True
elif response.status_code == 404:
    print(f"ℹ️  Instance '{INSTANCE_NAME}' does not exist yet")
    INSTANCE_EXISTS = False
else:
    print(f"⚠️  Error checking instance: {response.status_code}")
    print(response.text)
    INSTANCE_EXISTS = False

🔍 Checking if CloudSQL instance exists...
✅ Instance 'census-demo-db' already exists
   State: RUNNABLE
   Version: POSTGRES_15


In [6]:
# Create CloudSQL instance if it doesn't exist
if not INSTANCE_EXISTS:
    print(f"🚀 Creating CloudSQL instance '{INSTANCE_NAME}'...")
    print("⏱️  This will take approximately 10-15 minutes")
    
    create_url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances"
    
    instance_config = {
        "name": INSTANCE_NAME,
        "region": REGION,
        "databaseVersion": "POSTGRES_15",
        "settings": {
            "tier": "db-f1-micro",
            "dataDiskSizeGb": 10,
            "dataDiskType": "PD_SSD",
            "backupConfiguration": {
                "enabled": True,
                "startTime": "03:00"
            },
            "ipConfiguration": {
                "ipv4Enabled": True
            }
        },
        "rootPassword": DB_PASSWORD
    }
    
    response = requests.post(create_url, headers=headers, json=instance_config)
    
    if response.status_code in [200, 201]:
        print("✅ Instance creation started")
        operation = response.json()
        operation_name = operation.get('name')
        print(f"   Operation: {operation_name}")
        
        # Wait for operation to complete
        print("\n⏳ Waiting for instance creation to complete...")
        print("   (This typically takes 10-15 minutes for CloudSQL PostgreSQL)")
        max_wait_time = 1200  # 20 minutes
        start_time = time.time()
        last_status = None
        
        while time.time() - start_time < max_wait_time:
            op_url = f"https://sqladmin.googleapis.com/v1/{operation_name}"
            op_response = requests.get(op_url, headers=headers)
            
            if op_response.status_code == 200:
                op_data = op_response.json()
                status = op_data.get('status')
                
                if status == 'DONE':
                    if 'error' in op_data:
                        print(f"\n❌ Instance creation failed: {op_data['error']}")
                        break
                    else:
                        elapsed = int(time.time() - start_time)
                        print(f"\n✅ Instance created successfully! (took {elapsed}s)")
                        INSTANCE_EXISTS = True
                        break
                else:
                    elapsed = int(time.time() - start_time)
                    minutes = elapsed // 60
                    seconds = elapsed % 60
                    if status != last_status:
                        print(f"\n   Status: {status} - Elapsed: {minutes}m {seconds}s")
                        last_status = status
                    else:
                        print(f"   Status: {status} - Elapsed: {minutes}m {seconds}s", end='\r')
            
            time.sleep(15)  # Check every 15 seconds
        
        if not INSTANCE_EXISTS:
            elapsed = int(time.time() - start_time)
            print(f"\n⚠️  Instance creation timed out after {elapsed}s")
            print("   Please check the GCP Console to verify instance status.")
    else:
        print(f"❌ Failed to create instance: {response.status_code}")
        print(response.text)
else:
    print("ℹ️  Skipping instance creation - instance already exists")

ℹ️  Skipping instance creation - instance already exists


In [7]:
# Get instance connection details
print("📡 Fetching instance connection details...")

url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances/{INSTANCE_NAME}"
response = requests.get(url, headers=headers)

if response.status_code == 200:
    instance_info = response.json()
    connection_name = instance_info.get('connectionName')
    public_ip = instance_info['ipAddresses'][0]['ipAddress'] if instance_info.get('ipAddresses') else 'N/A'
    
    print("\n✅ Instance Information:")
    print(f"   Connection Name: {connection_name}")
    print(f"   Public IP: {public_ip}")
    print(f"   Database Version: {instance_info.get('databaseVersion')}")
    print(f"   State: {instance_info.get('state')}")
    print(f"   Region: {instance_info.get('region')}")
else:
    print(f"❌ Failed to get instance details: {response.status_code}")
    print(response.text)
    raise Exception("Cannot proceed without instance details")

📡 Fetching instance connection details...

✅ Instance Information:
   Connection Name: johnswain-1200-20260303091357:us-central1:census-demo-db
   Public IP: 34.41.181.208
   Database Version: POSTGRES_15
   State: RUNNABLE
   Region: us-central1


## Section 3: Connect to PostgreSQL and Create Database Schema

This section:
1. Establishes a connection to the CloudSQL instance
2. Creates the `census_data` database
3. Creates the `census_residence_type` table with appropriate schema
4. Adds indexes for query performance

In [8]:
# Initialize connector
print("🔌 Initializing Cloud SQL connector...")

# Ensure SSL environment variables are set (already done in Cell 5, but double-check)
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Create a new connector instance
# The connector uses these environment variables for its internal API calls
connector = Connector()

def getconn():
    conn = connector.connect(
        connection_name,
        "pg8000",
        user=DB_USER,
        password=DB_PASSWORD,
        db="postgres",  # Connect to default postgres database first
        enable_iam_auth=False
    )
    return conn

# Create engine for default postgres database
engine = sqlalchemy.create_engine(
    "postgresql+pg8000://",
    creator=getconn,
)

print("✅ Connector initialized")

🔌 Initializing Cloud SQL connector...
✅ Connector initialized


In [9]:
# Create census_data database if it doesn't exist
print(f"🗄️  Creating database '{DATABASE_NAME}'...")

try:
    with engine.connect() as conn:
        # Need to use autocommit mode to create database
        conn = conn.execution_options(isolation_level="AUTOCOMMIT")
        
        # Check if database exists
        result = conn.execute(text(
            f"SELECT 1 FROM pg_database WHERE datname = '{DATABASE_NAME}'"
        ))
        
        if result.fetchone():
            print(f"ℹ️  Database '{DATABASE_NAME}' already exists")
        else:
            conn.execute(text(f"CREATE DATABASE {DATABASE_NAME}"))
            print(f"✅ Database '{DATABASE_NAME}' created successfully")
except Exception as e:
    print(f"Error: {e}")
    # Database might already exist, continue
    pass

# Close the default connection
engine.dispose()
connector.close()

🗄️  Creating database 'census_data'...
✅ Database 'census_data' created successfully


In [10]:
# Reconnect to the census_data database
print(f"🔌 Connecting to database '{DATABASE_NAME}'...")

connector = Connector()

def getconn_census():
    conn = connector.connect(
        connection_name,
        "pg8000",
        user=DB_USER,
        password=DB_PASSWORD,
        db=DATABASE_NAME,
        enable_iam_auth=False
    )
    return conn

engine = sqlalchemy.create_engine(
    "postgresql+pg8000://",
    creator=getconn_census,
)

print("✅ Connected to census_data database")

🔌 Connecting to database 'census_data'...
✅ Connected to census_data database


In [11]:
# Create census_residence_type table
print("📊 Creating census_residence_type table...")

create_table_sql = """
CREATE TABLE IF NOT EXISTS census_residence_type (
    geography_code VARCHAR(20) PRIMARY KEY,
    date DATE,
    geography VARCHAR(255),
    geography_level VARCHAR(10),
    total_residents INTEGER,
    household_residents INTEGER,
    communal_residents INTEGER,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()
    print("✅ Table created successfully")

📊 Creating census_residence_type table...
✅ Table created successfully


In [12]:
# Create indexes for better query performance
print("🔍 Creating indexes...")

indexes = [
    "CREATE INDEX IF NOT EXISTS idx_geography_level ON census_residence_type(geography_level);",
    "CREATE INDEX IF NOT EXISTS idx_total_residents ON census_residence_type(total_residents);",
    "CREATE INDEX IF NOT EXISTS idx_geography ON census_residence_type(geography);"
]

with engine.connect() as conn:
    for idx_sql in indexes:
        conn.execute(text(idx_sql))
    conn.commit()
    print("✅ Indexes created successfully")

print("\n✅ Database schema setup complete!")

🔍 Creating indexes...
✅ Indexes created successfully

✅ Database schema setup complete!


## Section 4: Load Census Data from CSV Files

This section loads all 7 census CSV files into the PostgreSQL database:
- Output Areas (OA)
- Lower Super Output Areas (LSOA)
- Middle Super Output Areas (MSOA)
- Lower Tier Local Authorities (LTLA)
- Upper Tier Local Authorities (UTLA)
- Regions (RGN)
- Countries (CTRY)

In [13]:
# Define census data files and their geographic levels
census_files = [
    {"file": "census2021-ts001-oa.csv", "level": "OA", "name": "Output Areas"},
    {"file": "census2021-ts001-lsoa.csv", "level": "LSOA", "name": "Lower Super Output Areas"},
    {"file": "census2021-ts001-msoa.csv", "level": "MSOA", "name": "Middle Super Output Areas"},
    {"file": "census2021-ts001-ltla.csv", "level": "LTLA", "name": "Lower Tier Local Authorities"},
    {"file": "census2021-ts001-utla.csv", "level": "UTLA", "name": "Upper Tier Local Authorities"},
    {"file": "census2021-ts001-rgn.csv", "level": "RGN", "name": "Regions"},
    {"file": "census2021-ts001-ctry.csv", "level": "CTRY", "name": "Countries"}
]

# Base path to census data
data_path = Path("../source_data/census2021-ts001")

print(f"📂 Census data directory: {data_path.absolute()}")
print(f"   Directory exists: {data_path.exists()}")

📂 Census data directory: /Users/johnswain/Development/gcp-demo-dataplex/setup/../source_data/census2021-ts001
   Directory exists: True


In [14]:
# Load and insert data from all CSV files
print("📥 Loading census data into PostgreSQL...\n")

total_rows_loaded = 0
load_summary = []

for file_info in census_files:
    file_path = data_path / file_info["file"]
    level = file_info["level"]
    name = file_info["name"]
    
    print(f"📊 Processing {name} ({level})...")
    print(f"   File: {file_path.name}")
    
    if not file_path.exists():
        print(f"   ⚠️  File not found: {file_path}")
        continue
    
    try:
        # Read CSV file
        df = pd.read_csv(file_path)
        print(f"   Rows in CSV: {len(df):,}")
        
        # Rename columns to match database schema
        df = df.rename(columns={
            "date": "date",
            "geography": "geography",
            "geography code": "geography_code",
            "Residence type: Total; measures: Value": "total_residents",
            "Residence type: Lives in a household; measures: Value": "household_residents",
            "Residence type: Lives in a communal establishment; measures: Value": "communal_residents"
        })
        
        # Add geography_level column
        df["geography_level"] = level
        
        # Select only the columns we need
        df = df[[
            "geography_code", "date", "geography", "geography_level",
            "total_residents", "household_residents", "communal_residents"
        ]]
        
        # Convert date column to proper format
        df["date"] = pd.to_datetime(df["date"], format="%Y")
        
        # Insert data into database using pandas to_sql
        rows_inserted = df.to_sql(
            "census_residence_type",
            engine,
            if_exists="append",
            index=False,
            method="multi",
            chunksize=1000
        )
        
        total_rows_loaded += len(df)
        load_summary.append({
            "Level": level,
            "Name": name,
            "Rows": len(df)
        })
        
        print(f"   ✅ Inserted {len(df):,} rows\n")
        
    except Exception as e:
        print(f"   ❌ Error processing {file_path.name}: {e}\n")
        continue

print(f"\n{'='*60}")
print(f"✅ Data loading complete!")
print(f"   Total rows loaded: {total_rows_loaded:,}")
print(f"{'='*60}")

📥 Loading census data into PostgreSQL...

📊 Processing Output Areas (OA)...
   File: census2021-ts001-oa.csv
   Rows in CSV: 188,880
   ✅ Inserted 188,880 rows

📊 Processing Lower Super Output Areas (LSOA)...
   File: census2021-ts001-lsoa.csv
   Rows in CSV: 35,672
   ✅ Inserted 35,672 rows

📊 Processing Middle Super Output Areas (MSOA)...
   File: census2021-ts001-msoa.csv
   Rows in CSV: 7,264
   ✅ Inserted 7,264 rows

📊 Processing Lower Tier Local Authorities (LTLA)...
   File: census2021-ts001-ltla.csv
   Rows in CSV: 331
   ✅ Inserted 331 rows

📊 Processing Upper Tier Local Authorities (UTLA)...
   File: census2021-ts001-utla.csv
   Rows in CSV: 174
   ❌ Error processing census2021-ts001-utla.csv: (pg8000.exceptions.InterfaceError) in failed transaction block
(Background on this error at: https://sqlalche.me/e/20/rvf5)

📊 Processing Regions (RGN)...
   File: census2021-ts001-rgn.csv
   Rows in CSV: 10
   ✅ Inserted 10 rows

📊 Processing Countries (CTRY)...
   File: census2021-ts0

In [15]:
# Display load summary
summary_df = pd.DataFrame(load_summary)
summary_df["Rows"] = summary_df["Rows"].apply(lambda x: f"{x:,}")

print("\n📊 Load Summary by Geographic Level:\n")
print(summary_df.to_string(index=False))


📊 Load Summary by Geographic Level:

Level                         Name    Rows
   OA                 Output Areas 188,880
 LSOA     Lower Super Output Areas  35,672
 MSOA    Middle Super Output Areas   7,264
 LTLA Lower Tier Local Authorities     331
  RGN                      Regions      10


## Section 5: Validate and Query the Data

This section validates the data load and demonstrates various queries.

In [16]:
# Verify total row count
print("🔍 Validating data load...\n")

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM census_residence_type"))
    total_count = result.scalar()
    print(f"✅ Total rows in database: {total_count:,}")
    
    # Count by geography level
    result = conn.execute(text("""
        SELECT geography_level, COUNT(*) as count
        FROM census_residence_type
        GROUP BY geography_level
        ORDER BY count DESC
    """))
    
    print("\n📊 Row count by geographic level:")
    for row in result:
        print(f"   {row[0]:6s}: {row[1]:>8,} rows")

🔍 Validating data load...

✅ Total rows in database: 232,157

📊 Row count by geographic level:
   OA    :  188,880 rows
   LSOA  :   35,672 rows
   MSOA  :    7,264 rows
   LTLA  :      331 rows
   RGN   :       10 rows


In [17]:
# Display sample records from each geographic level
print("\n📄 Sample records from each geographic level:\n")

for file_info in census_files:
    level = file_info["level"]
    name = file_info["name"]
    
    query = f"""
        SELECT geography_code, geography, total_residents, 
               household_residents, communal_residents
        FROM census_residence_type
        WHERE geography_level = '{level}'
        LIMIT 3
    """
    
    df = pd.read_sql(query, engine)
    print(f"\n{name} ({level}):")
    print(df.to_string(index=False))


📄 Sample records from each geographic level:


Output Areas (OA):
geography_code geography  total_residents  household_residents  communal_residents
     E00060274 E00060274              273                  273                   0
     E00060275 E00060275              416                  416                   0
     E00060276 E00060276              259                  259                   0

Lower Super Output Areas (LSOA):
geography_code       geography  total_residents  household_residents  communal_residents
     E01011954 Hartlepool 001A             2284                 2284                   0
     E01011969 Hartlepool 001B             1344                 1344                   0
     E01011970 Hartlepool 001C             1070                 1070                   0

Middle Super Output Areas (MSOA):
geography_code      geography  total_residents  household_residents  communal_residents
     E02002483 Hartlepool 001            10239                10239                   0


In [ ]:
# Query: Top 10 areas by total residents
print("\n🏆 Top 10 areas by total residents:\n")

query = """
    SELECT geography, geography_level, total_residents, 
           household_residents, communal_residents
    FROM census_residence_type
    ORDER BY total_residents DESC
    LIMIT 10
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

In [ ]:
# Query: Summary statistics by geography level
print("\n📈 Summary statistics by geographic level:\n")

query = """
    SELECT 
        geography_level,
        COUNT(*) as num_areas,
        SUM(total_residents) as total_population,
        ROUND(AVG(total_residents), 0) as avg_residents,
        MIN(total_residents) as min_residents,
        MAX(total_residents) as max_residents,
        ROUND(AVG(CAST(communal_residents AS FLOAT) / NULLIF(total_residents, 0) * 100), 2) as pct_communal
    FROM census_residence_type
    GROUP BY geography_level
    ORDER BY num_areas DESC
"""

df = pd.read_sql(query, engine)
df["total_population"] = df["total_population"].apply(lambda x: f"{int(x):,}")
df["num_areas"] = df["num_areas"].apply(lambda x: f"{int(x):,}")
print(df.to_string(index=False))

In [ ]:
# Query: Areas with highest percentage of communal residents
print("\n🏘️  Top 10 areas by percentage of communal residents:\n")

query = """
    SELECT 
        geography,
        geography_level,
        total_residents,
        communal_residents,
        ROUND(CAST(communal_residents AS FLOAT) / NULLIF(total_residents, 0) * 100, 2) as pct_communal
    FROM census_residence_type
    WHERE communal_residents > 0 AND total_residents >= 100
    ORDER BY pct_communal DESC
    LIMIT 10
"""

df = pd.read_sql(query, engine)
print(df.to_string(index=False))

## Section 6: Enable Dataplex Universal Catalog Integration

This section enables Dataplex integration on the CloudSQL instance so that the database and table metadata will be automatically cataloged in Dataplex Universal Catalog.

**What this does:**
- Enables automatic metadata extraction from CloudSQL to Dataplex
- Allows discovery of your census_residence_type table in Dataplex search
- Enables enrichment with custom aspects and business metadata

**Note:** After enabling, it takes 2-48 hours for metadata to appear in Dataplex Universal Catalog (typically 2-6 hours for small instances).

In [ ]:
# Enable Dataplex integration on the CloudSQL instance
print("🔗 Enabling Dataplex Universal Catalog integration...")
print()

url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances/{INSTANCE_NAME}"
headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

patch_body = {
    "settings": {
        "enableDataplexIntegration": True
    }
}

try:
    response = requests.patch(url, headers=headers, json=patch_body)
    
    if response.status_code in [200, 201]:
        print("✅ Dataplex integration enabled successfully")
        operation = response.json()
        operation_name = operation.get('name')
        print(f"   Operation: {operation_name}")
        
        # Wait for operation to complete
        print("\n⏳ Waiting for operation to complete...")
        max_wait_time = 300  # 5 minutes
        start_time = time.time()
        
        while time.time() - start_time < max_wait_time:
            op_url = f"https://sqladmin.googleapis.com/v1/{operation_name}"
            op_response = requests.get(op_url, headers=headers)
            
            if op_response.status_code == 200:
                op_data = op_response.json()
                status = op_data.get('status')
                
                if status == 'DONE':
                    if 'error' in op_data:
                        print(f"\n❌ Operation failed: {op_data['error']}")
                        break
                    else:
                        elapsed = int(time.time() - start_time)
                        print(f"\n✅ Operation completed successfully! (took {elapsed}s)")
                        break
                else:
                    elapsed = int(time.time() - start_time)
                    print(f"   Status: {status} - Elapsed: {elapsed}s", end='\r')
            
            time.sleep(5)
        
        print()
        print("=" * 70)
        print("✅ Dataplex Integration Enabled!")
        print("=" * 70)
        print(f"   Instance: {INSTANCE_NAME}")
        print(f"   Database: {DATABASE_NAME}")
        print(f"   Table: census_residence_type")
        print()
        print("⏰ Next Steps:")
        print("   1. Wait 2-48 hours for metadata to sync to Dataplex Universal Catalog")
        print("   2. Search for your table at: https://console.cloud.google.com/dataplex/dp-search")
        print(f"   3. Search query: system=Cloud_SQL AND name=census_residence_type")
        print()
        print("💡 You can then enrich the cataloged table with:")
        print("   - Business descriptions")
        print("   - Custom aspects (data quality, PII classification, etc.)")
        print("   - Links to business glossary terms")
        print("=" * 70)
        
    else:
        print(f"❌ Failed to enable Dataplex integration: {response.status_code}")
        print(f"   Response: {response.text[:500]}")
        
except Exception as e:
    print(f"❌ Error enabling Dataplex integration: {e}")
    print("   You can enable it manually with:")
    print(f"   gcloud sql instances patch {INSTANCE_NAME} --enable-dataplex-integration --project={PROJECT_ID}")

In [ ]:
# Verify Dataplex integration is enabled
print("🔍 Verifying Dataplex integration status...")
print()

try:
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        instance_data = response.json()
        dataplex_enabled = instance_data.get('settings', {}).get('enableDataplexIntegration', False)
        
        if dataplex_enabled:
            print("✅ Verification successful!")
            print(f"   enableDataplexIntegration: {dataplex_enabled}")
            print()
            print("📊 Your CloudSQL resources will now be cataloged in Dataplex:")
            print(f"   - Instance: {INSTANCE_NAME}")
            print(f"   - Database: {DATABASE_NAME}")
            print(f"   - Table: census_residence_type")
            print(f"   - Columns: 8 columns (geography_code, date, geography, etc.)")
        else:
            print("⚠️  Dataplex integration is not enabled yet")
            print("   The operation may still be in progress")
    else:
        print(f"⚠️  Could not verify status: {response.status_code}")
        print(f"   Response: {response.text[:200]}")
        
except Exception as e:
    print(f"⚠️  Error verifying status: {e}")

## Section 7: Connection Information

Use this information to connect to the CloudSQL instance from external applications.

In [ ]:
# Display connection information
print("="*70)
print("📡 CloudSQL Connection Information")
print("="*70)
print(f"\nInstance Connection Name: {connection_name}")
print(f"Public IP Address: {public_ip}")
print(f"Database Name: {DATABASE_NAME}")
print(f"Database User: {DB_USER}")
print(f"\nPostgreSQL Connection String:")
print(f"postgresql://{DB_USER}:[PASSWORD]@{public_ip}:5432/{DATABASE_NAME}")
print(f"\npsql command:")
print(f"psql 'host={public_ip} port=5432 dbname={DATABASE_NAME} user={DB_USER}'")
print(f"\nCloud SQL Proxy command:")
print(f"cloud-sql-proxy {connection_name}")
print("="*70)
print("\n⚠️  Security Note:")
print("This instance is configured with public IP and allows connections from any IP.")
print("For production use, configure more restrictive authorized networks.")
print("="*70)

In [ ]:
# Close connections
print("\n🔌 Closing database connections...")
engine.dispose()
connector.close()
print("✅ Connections closed")

print("\n" + "="*70)
print("✅ Setup Complete!")
print("="*70)
print("\nYour CloudSQL PostgreSQL instance is now running with:")
print(f"  - {total_rows_loaded:,} census records across 7 geographic levels")
print(f"  - Database: {DATABASE_NAME}")
print(f"  - Table: census_residence_type")
print(f"  - Dataplex Universal Catalog integration: ENABLED")
print(f"\nThe database is ready to use and will remain active.")
print("\n📊 Dataplex Catalog:")
print(f"   Wait 2-48 hours for your table to appear in Dataplex search")
print(f"   Search at: https://console.cloud.google.com/dataplex/dp-search?project={PROJECT_ID}")
print("\n💡 Remember to delete the CloudSQL instance when no longer needed to avoid charges:")
print(f"   gcloud sql instances delete {INSTANCE_NAME} --project={PROJECT_ID}")
print("="*70)

## Section 8: Accessing Cloud SQL Assets in Dataplex Universal Catalog

After the metadata sync completes (2-48 hours), you can find and enrich your Cloud SQL assets in Dataplex.

### How to Search for Your Table

1. **Navigate to Dataplex Universal Catalog**:
   - Go to: https://console.cloud.google.com/dataplex/dp-search
   - Select "Dataplex Universal Catalog" as the search platform

2. **Search Options**:
   
   **Option A - Filter by System**:
   - In the Filters panel, click "Systems"
   - Select "Cloud SQL"
   - View all Cloud SQL assets
   
   **Option B - Search by Query**:
   - Enter: `system=Cloud_SQL AND type=Table`
   - Or: `system=Cloud_SQL AND name=census_residence_type`
   
   **Option C - Drill Down**:
   - Search for instance: `census-demo-db`
   - Click "Entry List" tab → "Show all children entries in search"
   - Select database: `census_data`
   - Click "Entry List" tab → "View child entries in search"
   - Find table: `census_residence_type`
   - Click "Schema" to view columns

### What Metadata is Cataloged

The following metadata is automatically extracted:

- **Instance Level**: census-demo-db (PostgreSQL 15, us-central1)
- **Database Level**: census_data
- **Table Level**: census_residence_type
  - Creation and modification dates
  - Row count and size
  - 8 columns with data types:
    - `geography_code` (VARCHAR)
    - `date` (DATE)
    - `geography` (VARCHAR)
    - `geography_level` (VARCHAR)
    - `total_residents` (INTEGER)
    - `household_residents` (INTEGER)
    - `communal_residents` (INTEGER)
    - `created_at` (TIMESTAMP)

### Enriching Metadata (Optional)

Once your table appears in Dataplex, you can enhance it:

#### 1. Add Business Description
Click "Overview" → "Add" to add a rich text description:
```
UK Census 2021 - Residence Type Data (TS001)

This table contains data on the number of usual residents in households 
and communal establishments from the 2021 Census for England and Wales.

Geographic Coverage: 232,157 rows across 7 geographic levels (OA, LSOA, 
MSOA, LTLA, UTLA, RGN, CTRY)

Source: UK Office for National Statistics
Release Date: 2022-12-13
```

#### 2. Attach Custom Aspects
Click "Aspects" → "Add" to attach metadata aspects:

- **Census Survey Metadata**:
  - Survey Year: 2021
  - Survey Type: Census
  - Geographic Coverage: England and Wales
  - Data Release Date: 2022-12-13

- **Data Quality**:
  - Total Records: 232,157
  - Data Completeness: 100%
  - Last Validated: [Current Date]

- **PII Classification**:
  - Contains PII: No
  - Data Sensitivity: Public
  - Classification Level: Unclassified

#### 3. Link to Business Glossary
If you've created glossary terms (via `apply_glossary_terms.ipynb`), you can link them to columns:
- Link "Population" terms to resident count columns
- Link "Geography" terms to geographic identifier columns

### Monitoring Sync Status

You can check when the metadata was last synced by viewing the entry details in Dataplex. The sync happens automatically every few hours after the initial sync completes.